In [13]:
import numpy as np
import pandas as pd
np.random.seed(42)

# input features
cgpa = np.random.uniform(low=5.0, high=10.0, size=2500)
iq = np.random.normal(loc=100, scale=15, size=2500)
marks_12th = np.random.uniform(low=60, high=100, size=2500)
marks_10th = np.random.uniform(low=60, high=100, size=2500)

# generating probabilities
placement_prob = (cgpa/10 * 0.3 + iq/150 * 0.3 + marks_12th/100 * 0.2 + marks_10th/100 * 0.2) # the floating point values are the weightage/importance of each feature in the overall probability 
placed = np.random.binomial(1, placement_prob)

# Create DataFrame
student_data = pd.DataFrame({
    'CGPA': np.round(cgpa, 2),
    'IQ': np.round(iq),
    'Marks_12th': np.round(marks_12th, 2),
    'Marks_10th': np.round(marks_10th, 2),
    'Placed': placed
})

student_data.head()

,CGPA,IQ,Marks_12th,Marks_10th,Placed
0,6.87,108.0,89.67,81.37,0
1,9.75,110.0,87.01,92.95,1
2,8.66,108.0,68.24,64.13,0
3,7.99,101.0,74.10,97.72,1
4,5.78,97.0,65.04,91.80,1


In [14]:
# Shape of the dataset
student_data.shape

(2500, 5)

In [15]:
from pandas.core.series import Series
from pandas.core.frame import DataFrame

# Splitting the data
X: DataFrame = student_data.drop(columns=['Placed'])
y: Series = student_data['Placed']

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class CustomDataset(Dataset):
    def __init__(self: CustomDataset, features: np.ndarray, labels: np.ndarray) -> None:
        scaler = StandardScaler() # global pre-processings and feature engineering techniques
        self.features = scaler.fit_transform(X = features) # applies feature wise (axis = 0)
        self.labels = labels
        
        self.features = torch.tensor(self.features, device='cpu', requires_grad=False)
        self.labels = torch.tensor(self.labels, device='cpu', requires_grad=False)

    def __len__(self: CustomDataset) -> int:
        return self.features.shape[0]

    def __getitem__(self: CustomDataset, index):
        return self.features[index], self.labels[index] # the shape of these tensors gets changed based on the type of data you are working with (tabular → (batch_size, no_features), image → (batch_size, no_channels, height, width), text → (batch_size, no_tokens, embedding_dimension))

In [17]:
# dataset object
dataset = CustomDataset(
    features=np.array(X),
    labels=np.array(y)
)

# length of dataset
len(dataset)

2500

In [18]:
# Accessing a row
index = int(input("Enter your index → "))
features, labels = dataset[index]

print(f"Features at index {index}:", features)
print(f"Labels at index {index}:", labels)

Features at index 3: tensor([ 0.3409,  0.0895, -0.4832,  1.5762], dtype=torch.float64)
Labels at index 3: tensor(1)


In [19]:
from torch.utils.data import BatchSampler, RandomSampler
sampler = RandomSampler(data_source=dataset)
batch_sampler = BatchSampler(sampler=sampler, batch_size=5, drop_last=False)

# Creating an object of dataset class → split the data into train and test then create a seperate dataloaders for each
dataloader = DataLoader(
    dataset=dataset,
    batch_size=5,
    # shuffle=True, # If True, the data is reshuffled at every epoch. False for data like `Time Series`
    sampler=sampler,
    batch_sampler=None,
    drop_last=False,
    pin_memory=True,
    num_workers=0 # if you mention more than zero workers then you will not be able to print the data
)

# Returning the generator
dataloader

##### sampler → 
A `sampler` defines the strategy to draw **individual samples** from the dataset. It yields one index at a time. It tells the DataLoader *which single item* to pick next <br>
`SequentialSampler`, `RandomSampler`, `WeightedRandomSampler` → Yields indices based on given probabilities (useful for imbalanced datasets).<br>
When you specify a `sampler` and a `batch_size`, the DataLoader will draw `batch_size` number of individual indices from the `sampler` and collate those individual items into a single batch.

##### batch_sampler →
A `batch_sampler` defines the strategy to draw a **whole batch of samples** at once. It yields a list (or batch) of indices at a time. It tells the DataLoader *which group of items* to pick next for a complete batch. <br>
<i>You use a `batch_sampler` when you need fine-grained control over which specific items end up in the same batch. For example, in natural language processing, you might want to group sentences of similar lengths into the same batch to minimize padding `custom function → collate_fn parameter`.</i><br>
If you specify a `batch_sampler`, you **cannot** specify `batch_size`, `shuffle`, `sampler`, or `drop_last`. The `batch_sampler` takes over the entire responsibility of batching and sampling.

In [20]:
for idx, (batch_features, batch_labels) in enumerate(dataloader): # At every iteration, dataloader return whatever you returned in __getitem__ function
    print(batch_features, batch_labels)
    if idx == 3:
        break
    print("_" * 50)

tensor([[-0.3538, -1.3752,  1.0323,  1.7296],
        [-0.8971,  0.2226, -0.1700,  0.9432],
        [ 1.7026, -0.7760, -1.0418,  0.4865],
        [ 1.2830,  1.2212, -0.5942, -1.2741],
        [-1.5160,  1.2212,  0.0052, -0.2408]], dtype=torch.float64) tensor([1, 0, 1, 1, 0])
__________________________________________________
tensor([[ 1.1390e-01,  1.2212e+00,  1.3047e+00, -1.0044e-03],
        [-1.5229e+00,  2.2876e-02, -1.0202e+00,  1.3300e-01],
        [-1.3579e+00, -1.0423e+00, -1.6751e+00,  1.1892e+00],
        [-1.5710e+00,  8.2179e-01, -6.8010e-01,  1.1460e+00],
        [ 1.2555e+00, -5.0973e-01,  1.1442e+00, -1.0334e+00]],
       dtype=torch.float64) tensor([0, 1, 1, 1, 1])
__________________________________________________
tensor([[-1.2960, -1.1755, -1.5736,  1.6926],
        [ 1.4000,  0.2892,  1.4374, -1.2035],
        [-0.0580,  1.5541, -1.0141, -0.7125],
        [ 0.7948, -1.1089,  1.3958,  0.7404],
        [-1.0071,  1.4876,  1.5667,  1.1874]], dtype=torch.float64) tensor(

---

TODO: Implement it yourself and Improvements

In [21]:
documents: list[str] = [
    "short",
    "a medium length document",
    "tiny",
    "this is a considerably longer document that we will use for testing",
    "another medium one",
    "x",
    "a somewhat long document here",
    "medium length text",
    "very short",
    "the longest document in this small dataset should be placed with others of similar size",
]

class DocumentDataset(Dataset):
    def __init__(self, documents):
        self.documents = documents
        vocabulary = sorted({word for document in documents for word in document.split()})
        self.vocab = {word: index + 1 for index, word in enumerate(vocabulary)}

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, idx):
        doc = self.documents[idx]
        tokens = torch.tensor([self.vocab[word] for word in doc.split()], dtype=torch.long)
        return tokens, len(tokens)

dataset = DocumentDataset(documents)
lengths: list[int] = [len(doc.split()) for doc in documents]
print(f"Lengths: {lengths}")

Lengths: [1, 4, 1, 12, 3, 1, 5, 3, 2, 15]


In [22]:
import random
from typing import Any
from collections import defaultdict
from torch.utils.data import Sampler


class LengthSortedBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, megabatch_multiplier=20, shuffle=True, seed=0, drop_last=False):
        # Initializing variables
        self.lengths: list[int] = list(lengths)
        self.batch_size = batch_size
        self.megabatch_size = batch_size * megabatch_multiplier
        self.shuffle = shuffle
        self.seed = seed
        self.drop_last = drop_last
        self.epoch = 0

    def set_epoch(self, epoch):
        """ we need a way for __iter__ to produce different random permutations on different calls, but still be deterministic if we rerun the same experiment. This is achieved by set_epoch method which is called during training loop and setting random seed """
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        indices: list[int] = list(range(len(self.lengths)))

        # Shuffling all the indices randomly
        if self.shuffle:
            rng.shuffle(indices)

        batches: list[Any] = []

        # Loop over each megabatch
        for start in range(0, len(indices), self.megabatch_size):
            # Forming a megabatch
            chunk: list[int] = indices[start : start + self.megabatch_size]

            # Sorting the documents based on document lengths in each megabatch
            chunk.sort(key=lambda i: self.lengths[i])

            # Loop over each document in each megabatch
            for j in range(0, len(chunk), self.batch_size):
                # Forming a batch
                batch: list[int] = chunk[j : j + self.batch_size]

                # If last batch in each megabatch have no. docs less than batch size then droping it if `drop_last = True` else keep that batch
                if self.drop_last and len(batch) < self.batch_size:
                    continue

                # Storing the batch (document indices)
                batches.append(batch)

        # Batch-wize shuffling
        if self.shuffle:
            rng.shuffle(batches)

        # Sampler must produce an iterator, not a list.
        return iter(batches)

    def __len__(self): # NOTE: The __len__ method isn't strictly required
        n: int = len(self.lengths)
        full, rem = divmod(n, self.megabatch_size)
        total = full * (self.megabatch_size // self.batch_size)

        if self.drop_last:
            total += rem // self.batch_size
        else:
            # If batches are perfectly formed without extra records then the formula is different than the presence of extra records
            total += (rem + self.batch_size - 1) // self.batch_size # from data structures and algorithms

        return total

In [23]:
def pad_collate(batch):
    sequences, sequence_lengths = zip(*batch)
    lengths = torch.tensor(sequence_lengths)
    padded = torch.nn.utils.rnn.pad_sequence(
        sequences, batch_first=True, padding_value=0
    )
    return padded, lengths

In [24]:
import torch.utils.data.dataloader
batch_sampler = LengthSortedBatchSampler(
    lengths, batch_size=4, drop_last=False
)

dataloader: torch.utils.data.dataloader.DataLoader[Any] = DataLoader(
    dataset=dataset,
    batch_sampler=batch_sampler,
    collate_fn=pad_collate
)

for idx, (batch_features, doc_lengths) in enumerate(dataloader): # At every iteration, dataloader return whatever you returned in __getitem__ function
    print(batch_features, doc_lengths)
    if idx != len(dataloader) - 1:
        print("_" * 50)

tensor([[30, 10,  1,  4, 13,  6, 28, 34, 35, 32,  7, 26,  0,  0,  0],
        [29, 14,  6,  9, 30, 24,  5, 21,  3, 19, 36, 18, 16, 22, 23]]) tensor([12, 15])
__________________________________________________
tensor([[37,  0],
        [31,  0],
        [20,  0],
        [33, 20]]) tensor([1, 1, 1, 2])
__________________________________________________
tensor([[15, 11, 27,  0,  0],
        [ 2, 15, 17,  0,  0],
        [ 1, 15, 11,  6,  0],
        [ 1, 25, 12,  6,  8]]) tensor([3, 3, 4, 5])


The sampler decides which indices go together in a batch; the collate_fn is an operation appled to all the records in a batch and across the batch and return the same structure of output as provided input.